# Malaysia Weather Data Cleaning Process

This notebook documents the cleaning process used to prepare the Malaysia weather dataset for analysis. The aim is to keep the workflow short, transparent, and easy to verify.

The process has five main steps:

1. Load and inspect the raw and cleaned datasets.
2. Remove rows where all weather measurement fields are missing.
3. Fill the remaining missing weather values while keeping observed values unchanged.
4. Create useful time features: `month_number` and `datetime`.
5. Validate the final cleaned dataset.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

RAW_DATA_PATH = Path(r'/Users/c2/Desktop/malaysia_weather_data.csv')
CLEANED_DATA_PATH = Path(r'/Users/c2/Library/CloudStorage/GoogleDrive-zche0400@student.monash.edu/My Drive/ADS1001/project/MWF-G2/data/malaysia_weather_cleaned_file.csv')

weather_cols = [
    'temperature', 'pressure', 'dew_point', 'humidity', 'wind_speed',
    'gust', 'wind_chill', 'uv_index', 'precipitation_rate', 'precipitation_total'
]

time_cols = ['year', 'month', 'day', 'hour', 'minutes', 'seconds']
location_cols = ['place', 'city', 'state']
key_cols = location_cols + time_cols

raw_df = pd.read_csv(RAW_DATA_PATH)
cleaned_df = pd.read_csv(CLEANED_DATA_PATH)

print(f'Raw dataset shape: {raw_df.shape}')
print(f'Cleaned dataset shape: {cleaned_df.shape}')

Raw dataset shape: (57475, 19)
Cleaned dataset shape: (51693, 22)


## 1. Initial Data Audit

The raw dataset contained weather observations by location and timestamp. The first check focused on row count, column count, duplicate rows, data types, and missing values.

In [2]:
def dataset_overview(df: pd.DataFrame, name: str) -> pd.DataFrame:
    return pd.DataFrame({
        'dataset': [name],
        'rows': [len(df)],
        'columns': [df.shape[1]],
        'duplicate_rows': [df.duplicated().sum()],
        'total_missing_values': [df.isna().sum().sum()]
    })

summary = pd.concat([
    dataset_overview(raw_df, 'Raw data'),
    dataset_overview(cleaned_df, 'Cleaned data')
], ignore_index=True)

display(summary)

dtype_comparison = pd.DataFrame({
    'raw_dtype': raw_df.dtypes.astype(str),
    'cleaned_dtype': cleaned_df.reindex(columns=raw_df.columns).dtypes.astype(str)
})

display(dtype_comparison)

,dataset,rows,columns,duplicate_rows,total_missing_values
0,Raw data,57475,19,0,111093
1,Cleaned data,51693,22,0,51693


,raw_dtype,cleaned_dtype
place,object,object
city,object,object
state,object,object
temperature,float64,float64
pressure,float64,float64
dew_point,float64,float64
humidity,float64,float64
wind_speed,float64,float64
gust,float64,float64
wind_chill,float64,float64


In [3]:
missing_before = (
    raw_df[weather_cols]
    .isna()
    .sum()
    .rename('missing_count')
    .to_frame()
)
missing_before['missing_percent'] = missing_before['missing_count'] / len(raw_df) * 100

display(missing_before.sort_values('missing_count', ascending=False))

,missing_count,missing_percent
uv_index,18298,31.836
gust,13284,23.113
precipitation_rate,10901,18.967
precipitation_total,10901,18.967
dew_point,10510,18.286
wind_chill,10150,17.660
temperature,10112,17.594
humidity,10105,17.582
wind_speed,8642,15.036
pressure,8190,14.250


## 2. Remove Rows With No Weather Measurements

Some rows had no usable weather measurement at all. These rows were removed because they could not support weather analysis or reliable imputation.

In [4]:
all_weather_missing = raw_df[weather_cols].isna().all(axis=1)
raw_after_row_filter = raw_df.loc[~all_weather_missing].reset_index(drop=True)
removed_rows = raw_df.loc[all_weather_missing]

row_filter_summary = pd.DataFrame({
    'metric': [
        'Rows in raw data',
        'Rows removed because all weather measurements were missing',
        'Rows after removing unusable records',
        'Rows in cleaned data'
    ],
    'value': [
        len(raw_df),
        int(all_weather_missing.sum()),
        len(raw_after_row_filter),
        len(cleaned_df)
    ]
})

display(row_filter_summary)

display(removed_rows[weather_cols].isna().sum().rename('missing_values_in_removed_rows').to_frame())

,metric,value
0,Rows in raw data,57475
1,Rows removed because all weather measurements ...,5782
2,Rows after removing unusable records,51693
3,Rows in cleaned data,51693


,missing_values_in_removed_rows
temperature,5782
pressure,5782
dew_point,5782
humidity,5782
wind_speed,5782
gust,5782
wind_chill,5782
uv_index,5782
precipitation_rate,5782
precipitation_total,5782


## 3. Fill Remaining Missing Weather Values

After removing unusable rows, the remaining dataset still had missing values in individual weather columns. These missing cells were filled so that later analysis could use a complete set of weather measurements.

The validation below compares the filtered raw data with the cleaned data. It confirms that observed values were preserved and only originally missing cells were filled.

In [5]:
# Check whether the row order and identifying columns still match after the row filter.
key_order_matches = raw_after_row_filter[key_cols].equals(cleaned_df[key_cols])
print(f'Key columns match after row filtering: {key_order_matches}')

imputation_checks = []
for col in weather_cols:
    raw_col = raw_after_row_filter[col]
    clean_col = cleaned_df[col]

    missing_before = raw_col.isna().sum()
    missing_after = clean_col.isna().sum()

    observed_mask = raw_col.notna()
    observed_values_changed = (raw_col[observed_mask].astype(str) != clean_col[observed_mask].astype(str)).sum()

    imputation_checks.append({
        'column': col,
        'missing_before_imputation': int(missing_before),
        'missing_after_cleaning': int(missing_after),
        'observed_values_changed': int(observed_values_changed)
    })

imputation_checks = pd.DataFrame(imputation_checks)
display(imputation_checks)

Key columns match after row filtering: True


,column,missing_before_imputation,missing_after_cleaning,observed_values_changed
0,temperature,4330,0,0
1,pressure,2408,0,0
2,dew_point,4728,0,0
3,humidity,4323,0,0
4,wind_speed,2860,0,0
5,gust,7502,0,0
6,wind_chill,4368,0,0
7,uv_index,12516,0,0
8,precipitation_rate,5119,0,0
9,precipitation_total,5119,0,0


## 4. Create Time Features

The raw data stored time across separate columns. The cleaned data adds:

- `month_number`: numeric month value mapped from the three-letter month abbreviation.
- `datetime`: a single timestamp built from year, month, day, hour, minute, and second.

In [6]:
month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

month_mapping_table = (
    cleaned_df[['month', 'month_number']]
    .drop_duplicates()
    .sort_values('month_number')
    .reset_index(drop=True)
)

display(month_mapping_table)

expected_month_number = cleaned_df['month'].map(month_map)
month_number_mismatches = (expected_month_number != cleaned_df['month_number']).sum()

expected_datetime_input = cleaned_df[['year', 'month_number', 'day', 'hour', 'minutes', 'seconds']].rename(
    columns={'month_number': 'month', 'minutes': 'minute', 'seconds': 'second'}
)
expected_datetime = pd.to_datetime(expected_datetime_input, errors='coerce')
actual_datetime = pd.to_datetime(cleaned_df['datetime'], errors='coerce')
datetime_mismatches = (expected_datetime != actual_datetime).sum()

feature_validation = pd.DataFrame({
    'check': ['month_number mismatches', 'datetime parse errors', 'datetime mismatches'],
    'value': [
        int(month_number_mismatches),
        int(actual_datetime.isna().sum()),
        int(datetime_mismatches)
    ]
})

display(feature_validation)

,month,month_number
0,Jan,1
1,Feb,2
2,Mar,3
3,Apr,4
4,May,5
5,Jun,6
6,Jul,7
7,Aug,8
8,Sep,9
9,Oct,10


,check,value
0,month_number mismatches,0
1,datetime parse errors,0
2,datetime mismatches,0


## 5. Add Air Quality Placeholder

The cleaned dataset includes an `air_quality` column. It is currently empty, so it should be treated as a placeholder for future data integration rather than an active analysis variable.

In [7]:
air_quality_summary = pd.DataFrame({
    'metric': ['Rows', 'Missing air_quality values', 'Non-missing air_quality values'],
    'value': [
        len(cleaned_df),
        int(cleaned_df['air_quality'].isna().sum()),
        int(cleaned_df['air_quality'].notna().sum())
    ]
})

display(air_quality_summary)

,metric,value
0,Rows,51693
1,Missing air_quality values,51693
2,Non-missing air_quality values,0


## 6. Final Validation

The final checks confirm that the cleaned dataset is ready for analysis: no duplicate rows, no missing values in the main weather measurement columns, and valid timestamp values.

In [8]:
added_cols = [col for col in cleaned_df.columns if col not in raw_df.columns]

final_validation = pd.DataFrame({
    'metric': [
        'Final rows',
        'Final columns',
        'Duplicate rows',
        'Missing values in weather measurement columns',
        'Invalid datetime values',
        'Added columns'
    ],
    'value': [
        len(cleaned_df),
        cleaned_df.shape[1],
        int(cleaned_df.duplicated().sum()),
        int(cleaned_df[weather_cols].isna().sum().sum()),
        int(pd.to_datetime(cleaned_df['datetime'], errors='coerce').isna().sum()),
        ', '.join(added_cols)
    ]
})

display(final_validation)

,metric,value
0,Final rows,51693
1,Final columns,22
2,Duplicate rows,0
3,Missing values in weather measurement columns,0
4,Invalid datetime values,0
5,Added columns,"month_number, datetime, air_quality"


## Cleaning Summary

The raw dataset started with **57,475 rows** and **19 columns**. The cleaned dataset has **51,693 rows** and **22 columns**.

The cleaning process removed **5,782 rows** where all weather measurement fields were missing. The remaining missing weather values were filled, while original observed values were kept unchanged. The cleaned file also adds `month_number`, `datetime`, and `air_quality` to support later analysis and future data integration.